# Análise de Clusters (K-Means) — Vulnerabilidade Habitacional por Setor Censitário
## São Paulo (Capital) — Censo Demográfico 2022 (IBGE)

Este notebook replica, para o município de São Paulo, a metodologia desenvolvida na análise de Araraquara: identificação de padrões de precariedade habitacional e infraestrutural por setor censitário, usando dados do Censo 2022 (IBGE) e o algoritmo K-Means.

Diferente de Araraquara (cidade de médio porte, onde a maioria das variáveis de privação extrema era zero), São Paulo é uma metrópole — logo o conjunto de variáveis ativas (não-zero) tende a ser bem mais amplo. Por isso, a seleção final de variáveis é refeita do zero a partir da correlação de Pearson nas 644 variáveis originais, e não reaproveitada diretamente de Araraquara.


## 1. Carregamento dos Dados Consolidados

In [ ]:
from pathlib import Path
import pandas as pd
import re

# Caminho relativo: a pasta 'data' deve estar no mesmo nível deste notebook
caminho = Path("data") / "setores_censitarios_sp_capital.csv"

df = pd.read_csv(
    caminho,
    sep=";",
    encoding="latin1",
    dtype={"cd_setor": str, "cd_municipio": str, "CD_SETOR": str, "CD_MUN": str}
)

print("Dataset carregado com sucesso!")
print(f" Total de setores (linhas): {df.shape[0]}")
print(f" Total de variáveis (colunas): {df.shape[1]}")
df.head()


### 1.1 Limpeza Numérica

O IBGE aplica **sigilo estatístico** em setores muito pequenos (poucos domicílios): em vez do valor real, a célula recebe o marcador de texto `'X'`. Além disso, colunas de taxa/média vindas da tabela Básico (`v0005`, por exemplo) usam **vírgula como separador decimal** (padrão brasileiro), o que faz o pandas ler a coluna inteira como texto.

A função abaixo resolve os dois problemas de uma vez, em todas as colunas de variáveis do Censo (`V00001`...`V00643` e `v0001`...`v0009`): troca vírgula por ponto, converte para número, e qualquer valor que não vire número (como o `'X'` de sigilo) é tratado como `0`.


In [ ]:
def limpar_numero(serie):
    """Converte uma coluna de texto (com vírgula decimal e/ou marcadores
    de sigilo como 'X') para número. Valores não conversíveis viram 0."""
    serie_limpa = serie.astype(str).str.replace(",", ".", regex=False)
    return pd.to_numeric(serie_limpa, errors="coerce").fillna(0)

# Identifica todas as colunas de variáveis do Censo (V00xxx e v0xxx)
padrao_variavel = re.compile(r"^[Vv]0*\d+$")
colunas_variaveis = [c for c in df.columns if padrao_variavel.match(c)]

print(f"Aplicando limpeza numérica em {len(colunas_variaveis)} colunas...")
for col in colunas_variaveis:
    df[col] = limpar_numero(df[col])

print("Limpeza concluída!")


### 1.2 Remoção de Colunas Administrativas Não Utilizadas

Seguindo o mesmo critério de limpeza aplicado na base de Araraquara: removemos colunas geográficas/administrativas que não entram na análise (região, núcleo urbano, RGI, área em km² etc.) e denominadores não utilizados (`v0001` a `v0004`, `v0006`).

**Mantemos** `CD_FCU`/`NM_FCU` (necessárias para a variável de favela), `CD_MUN`/`NM_MUN` (identificação geográfica), `v0005`, `v0007`, `v0008` e `v0009`.


In [ ]:
columns_to_remove = [
    "CD_REGIAO", "NM_REGIAO",
    "CD_NU", "NM_NU",
    "CD_AGLOM", "NM_AGLOM",
    "CD_RGINT", "NM_RGINT",
    "CD_RGI", "NM_RGI",
    "CD_CONCURB", "NM_CONCURB",
    "AREA_KM2",
    "v0001", "v0002", "v0003", "v0004", "v0006",
]

antes = df.shape[1]
df = df.drop(columns=columns_to_remove, errors="ignore")
print(f"Colunas removidas: {antes - df.shape[1]} (de {antes} para {df.shape[1]})")


## 2. Criação da Variável de População em Favela (N_POP_DPPO)

O Censo 2022 identifica, na própria tabela Básico, os setores que pertencem a uma Favela ou Comunidade Urbana através da coluna `CD_FCU`. Atenção a duas particularidades da forma como o IBGE preenche essa coluna:

- Quando o setor **não** pertence a nenhuma FCU, o IBGE **não deixa a célula vazia (NaN)** — ele preenche com o caractere `'.'` (marcador de "não aplicável"). Por isso, checar apenas `.notna()` não basta.
- A variável de população correta aqui é **`V00005`** (maiúsculo, contagem total de moradores em domicílios permanentes, vinda da tabela de Domicílios) — não confundir com `v0005` (minúsculo, que na tabela Básico é a **média** de moradores por domicílio, não uma contagem).

Criamos `N_POP_DPPO`: a população total (V00005) atribuída apenas aos setores classificados como pertencentes a uma favela/comunidade urbana; nos demais setores, o valor é 0.


In [ ]:
# Setor pertence a uma Favela/Comunidade Urbana quando CD_FCU tem um código real
# (diferente de vazio E diferente de '.' — marcador do IBGE para "não aplicável")
df["em_favela"] = df["CD_FCU"].notna() & (df["CD_FCU"].astype(str).str.strip() != ".")

# População em favela = V00005 (maiúsculo!) apenas nos setores marcados como favela
df["N_POP_DPPO"] = df["V00005"].where(df["em_favela"], 0)

print(f"Setores classificados como Favela/Comunidade Urbana: {df['em_favela'].sum()} de {len(df)}")
print(f"População total em Favelas/Comunidades Urbanas: {df['N_POP_DPPO'].sum():,.0f} moradores")


## 3. Estatística Descritiva Geral

Assim como na análise de Araraquara, o primeiro passo é examinar todas as variáveis numéricas sem qualquer filtro, para entender a distribuição geral dos dados (médias, mínimos, máximos, nulos e zeros).


In [ ]:
import numpy as np

df_num = df.select_dtypes(include=[np.number])

estatistica = df_num.describe().T
estatistica["qtd_nulos"] = df_num.isnull().sum()
estatistica["qtd_zeros"] = (df_num == 0).sum()

display(
    estatistica[["count", "mean", "min", "max", "qtd_nulos", "qtd_zeros"]].round(2)
)


## 4. Análise de Correlação Inicial (644 Variáveis Originais)

Repetindo a metodologia de Araraquara: aplicamos o teste de correlação de Pearson em todas as variáveis numéricas originais para identificar redundâncias e variáveis "espelhadas" antes de qualquer seleção manual.


In [ ]:
# 1. Calcula a matriz de correlação
correlation_matrix = df_num.corr(method="pearson")

# 2. Converte a matriz em lista de pares
df_pairs = (
    correlation_matrix.where(
        np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
    )
    .stack()
    .reset_index()
)
df_pairs.columns = ["Variable A", "Variable B", "Correlation"]

# 3. Filtra correlações fortes
threshold = 0.85
df_screening = df_pairs[df_pairs["Correlation"].abs() >= threshold].copy()

# 4. Define o tipo de relação
df_screening["Relationship Type"] = np.where(
    df_screening["Correlation"] >= threshold,
    "Direct (Same Direction)",
    "Inverse (Mirror)",
)

df_screening = df_screening.sort_values(by="Correlation", ascending=False).round(2)
df_screening.to_csv(
    "initial_variable_correlation_analysis_sp.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False,
)

print(f"Total de pares com correlação forte (|r| >= {threshold}): {len(df_screening)}")
display(df_screening.head(20))


## 5. Seleção de Variáveis — Base Teórica (SDG 11.1)

Partindo da mesma fundamentação conceitual usada em Araraquara (privação absoluta, conforme SDG Target 11.1 da ONU), mantemos a lista de 31 variáveis "puras" de privação, somadas à variável de favela (`N_POP_DPPO`) criada acima — totalizando **32 variáveis candidatas**.

Diferente de Araraquara, **não vamos excluir variáveis manualmente aqui**: como São Paulo é uma metrópole, é esperado que a maioria dessas variáveis tenha valores ativos. A seleção final das variáveis "vivas" (não constantes em zero) é feita automaticamente pelo código na próxima etapa — evitando qualquer viés de reaproveitar a lista final de Araraquara, que foi calibrada para uma realidade de cidade muito menor.


In [ ]:
# 1. Lista base de variáveis de privação (mesma fundamentação teórica de Araraquara)
pre_selected_variables = [
    "V00005", "V00006", "V00008", "V00009", "V00050", "V00051",
    "V00052", "V00053", "V00054", "V00055", "V00056", "V00057",
    "V00058", "V00113", "V00114", "V00115", "V00116", "V00117",
    "V00118", "V00200", "V00201", "V00464", "V00236", "V00237",
    "V00312", "V00313", "V00314", "V00315", "V00316", "V00399",
    "V00400", "V00401", "V00402",
    "N_POP_DPPO",  # variável adicional: população em favela/comunidade urbana
]

# 2. Garante conversão numérica
vars_existentes = [c for c in pre_selected_variables if c in df.columns]
df_num_pre = df[vars_existentes].apply(pd.to_numeric, errors="coerce").fillna(0)

# 3. Diagnóstico: quais variáveis têm dados reais (valor > 0) em São Paulo?
print("=== VARIÁVEIS COM DADOS (VALOR > 0) EM SÃO PAULO ===")
nao_zeradas = (df_num_pre > 0).sum()
print(nao_zeradas[nao_zeradas > 0].sort_values(ascending=False))

print("\n=== VARIÁVEIS 100% ZERADAS EM SÃO PAULO ===")
zeradas = nao_zeradas[nao_zeradas == 0].index.tolist()
print(zeradas if zeradas else "Nenhuma — todas as variáveis têm ao menos um valor ativo.")


**Compare esse resultado com Araraquara**: lá, 21 das 31 variáveis eram 100% zeradas. É esperado que em São Paulo o número de variáveis zeradas seja bem menor (ou até zero), justamente porque a metrópole concentra formas de precariedade habitacional (cortiços, favelas, estruturas improvisadas) praticamente ausentes em cidades médias do interior.

## 6. Seleção Final das Variáveis Ativas (Automática)

Diferente de Araraquara (onde a lista final de 10 variáveis foi definida manualmente, pois era um conjunto pequeno e óbvio), em São Paulo a lista de variáveis ativas tende a ser maior — por isso, o filtro abaixo é feito de forma **automática e reprodutível**: mantemos toda variável que teve ao menos um valor diferente de zero em algum setor de São Paulo.


In [ ]:
# Seleção automática: mantém apenas variáveis com pelo menos 1 valor > 0
filtered_variables = nao_zeradas[nao_zeradas > 0].index.tolist()

print(f"Total de variáveis ativas selecionadas para São Paulo: {len(filtered_variables)}")
print(filtered_variables)

df_num_filtered = df[filtered_variables].apply(pd.to_numeric, errors="coerce").fillna(0)


## 7. Matriz de Correlação e Heatmap — Variáveis Ativas (Dados Brutos)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

correlation_matrix = df_num_filtered.corr(method="pearson")

df_pairs = (
    correlation_matrix.where(
        np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
    )
    .stack()
    .reset_index()
)
df_pairs.columns = ["Variable A", "Variable B", "Correlation"]
df_pairs = df_pairs.sort_values(by="Correlation", ascending=False)

txt_output = "raw_variables_correlation_list_sp.txt"
with open(txt_output, "w", encoding="utf-8") as f:
    f.write("=== RAW VARIABLES CORRELATION LIST (São Paulo) ===\n\n")
    f.write(df_pairs.to_string(index=False, formatters={"Correlation": "{:.4f}".format}))

print(f"TXT salvo em: {txt_output}")

plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(
    correlation_matrix,
    mask=mask,
    cmap="coolwarm",
    vmax=1.0,
    vmin=-1.0,
    center=0,
    square=True,
    linewidths=0.5,
    annot=True,
    fmt=".2f",
    cbar_kws={"shrink": 0.8, "label": "Pearson Correlation Coefficient"},
    annot_kws={"size": 7},
)

plt.title("Pearson Correlation Matrix - Raw Census Variables (São Paulo)", fontsize=14, fontweight="bold", pad=15)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()

plt.savefig("raw_variables_correlation_heatmap_sp.png", dpi=300, bbox_inches="tight")
plt.show()


**Ponto de atenção (mesmo efeito visto em Araraquara — o "Efeito de Escala Demográfica"):** nas variáveis brutas (contagens absolutas), setores maiores/mais populosos tendem a mostrar correlações artificialmente altas entre si, simplesmente porque setores com mais domicílios têm mais de tudo. Isso será corrigido no próximo passo, convertendo os valores absolutos em taxas percentuais (dividindo pelo denominador `v0007`).

## 8. Transformação em Taxas Percentuais

Seguindo a mesma lógica metodológica de Araraquara: as variáveis de privação domiciliar são divididas por `v0007` (Total de Domicílios Particulares Ocupados), e as variáveis demográficas (moradores, crianças) são tratadas com o denominador populacional adequado (`v0005`), neutralizando o efeito do tamanho do setor.


In [ ]:
# 1. Separa variáveis com denominador especial (não usam v0007 como base)
vars_denominador_especial = [v for v in ["V00005", "V00006", "N_POP_DPPO"] if v in filtered_variables]
vars_domiciliares = [v for v in filtered_variables if v not in vars_denominador_especial]

# 2. Calcula as taxas por domicílio (% em relação a v0007) para as variáveis "padrão"
df_rates = df[vars_domiciliares].div(df["v0007"].replace(0, np.nan), axis=0) * 100
df_rates.columns = [f"{var}_P" for var in vars_domiciliares]
df_rates = df_rates.fillna(0)

# 3. Casos especiais demográficos (proporção de crianças, moradores por domicílio, etc.)
if "V00008" in filtered_variables and "V00005" in df.columns:
    df_rates["V00008_P"] = (df["V00008"] / df["V00005"].replace(0, np.nan)) * 100
if "V00009" in filtered_variables and "V00006" in df.columns:
    df_rates["V00009_P"] = (df["V00009"] / df["V00006"].replace(0, np.nan)) * 100

# 4. N_POP_DPPO: proporção da população do setor que vive em favela/comunidade urbana
#    (será sempre 0% ou 100%, pois a classificação é por setor inteiro)
if "N_POP_DPPO" in filtered_variables and "V00005" in df.columns:
    df_rates["N_POP_DPPO_P"] = (df["N_POP_DPPO"] / df["V00005"].replace(0, np.nan)) * 100

df_rates = df_rates.fillna(0)

# 4. Insere identificadores geográficos no início
cols_geo = ["cd_setor", "CD_SETOR", "cd_municipio", "CD_MUN", "NM_MUN", "NM_UF", "CD_UF"]
for col in reversed(cols_geo):
    if col in df.columns and col not in df_rates.columns:
        df_rates.insert(0, col, df[col].values)

# 5. Estatística descritiva das taxas
rates_descriptive = (
    df_rates.select_dtypes(include="number")
    .describe()
    .T[["count", "mean", "std", "min", "max"]]
    .rename(columns={
        "count": "Count", "mean": "Mean (%)",
        "std": "Standard Deviation", "min": "Minimum (%)", "max": "Maximum (%)"
    })
    .round(2)
)

txt_output = "selected_variables_rates_statistics_table_sp.txt"
with open(txt_output, "w", encoding="utf-8") as f:
    f.write("DESCRIPTIVE STATISTICS TABLE (PROPORTIONS / RATES) - São Paulo\n\n")
    f.write(rates_descriptive.to_string())

print(f"Tabela salva em: {txt_output}")
display(rates_descriptive)


**Checagem final de variância zero nas taxas**: assim como aconteceu com `V00058_P` em Araraquara, é possível que alguma taxa fique 100% zerada mesmo em São Paulo. O código abaixo identifica e remove automaticamente essas colunas antes da normalização (evita erro de divisão por zero no z-score / MinMax).

In [ ]:
rate_columns_all = [c for c in df_rates.columns if c.endswith("_P")]

variancia_zero = df_rates[rate_columns_all].std() == 0
colunas_a_remover = variancia_zero[variancia_zero].index.tolist()

if colunas_a_remover:
    print(f"Removendo variáveis com variância zero: {colunas_a_remover}")
    df_rates = df_rates.drop(columns=colunas_a_remover)
else:
    print("Nenhuma variável com variância zero encontrada — todas serão mantidas.")

output_file = "df_rates_sp.csv"
df_rates.to_csv(output_file, sep=";", index=False, encoding="utf-8-sig")
print(f"Arquivo de taxas salvo em: {output_file}")


## 9. Normalização Min-Max (0 a 1)

Última etapa antes do K-Means: normalizar as taxas para a mesma escala (0 a 1), garantindo que nenhuma variável domine o modelo apenas por ter uma magnitude naturalmente maior.


In [ ]:
from sklearn.preprocessing import MinMaxScaler

rate_columns = [c for c in df_rates.columns if c.endswith("_P")]
geo_cols = [c for c in ["cd_setor", "CD_SETOR", "cd_municipio", "CD_MUN", "NM_MUN", "NM_UF", "CD_UF"] if c in df_rates.columns]

scaler = MinMaxScaler()
df_rates_normalized = pd.DataFrame(
    scaler.fit_transform(df_rates[rate_columns]),
    columns=rate_columns,
    index=df_rates.index,
)

for col in reversed(geo_cols):
    df_rates_normalized.insert(0, col, df_rates[col].values)

output_file = "df_rates_normalized_sp.csv"
df_rates_normalized.to_csv(output_file, sep=";", index=False, encoding="utf-8-sig")

print(f"Normalização (MinMax 0-1) concluída para {len(rate_columns)} variáveis!")
print(f"Arquivo salvo em: {output_file}\n")
display(df_rates_normalized.head())


## 10. Seleção do Número de Clusters (K): Método do Cotovelo + Silhouette

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df_normalized = pd.read_csv("df_rates_normalized_sp.csv", sep=";")

feature_cols = [c for c in df_normalized.columns if c.endswith("_P")]
X = df_normalized[feature_cols].fillna(0)

k_range = range(2, 11)
models = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(X) for k in k_range]

inertias = [model.inertia_ for model in models]
silhouette_scores = [silhouette_score(X, model.labels_) for model in models]

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.set_xlabel("Número de Clusters (K)", fontsize=11, fontweight="bold")
ax1.set_ylabel("Inércia (Soma dos Erros Quadráticos)", color="tab:blue", fontsize=11, fontweight="bold")
ax1.plot(k_range, inertias, marker="o", color="tab:blue", linewidth=2)
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.grid(True, linestyle="--", alpha=0.5)

ax2 = ax1.twinx()
ax2.set_ylabel("Coeficiente de Silhouette", color="tab:red", fontsize=11, fontweight="bold")
ax2.plot(k_range, silhouette_scores, marker="s", color="tab:red", linewidth=2, linestyle="--")
ax2.tick_params(axis="y", labelcolor="tab:red")

plt.title("Seleção do Número de Clusters (K): Método do Cotovelo vs. Silhouette Score\n(São Paulo - Censo 2022)", fontsize=12, fontweight="bold", pad=12)
fig.tight_layout()

plt.savefig("elbow_silhouette_selection_plot_sp.png", dpi=300, bbox_inches="tight")
plt.show()

df_metrics = pd.DataFrame({
    "K (Clusters)": list(k_range),
    "Inércia": inertias,
    "Silhouette Score": silhouette_scores
}).round(4)

display(df_metrics)


**Como escolher o K**: procure o "cotovelo" na curva azul (ponto onde a inércia para de cair rapidamente) e o pico (ou platô alto) na curva vermelha de Silhouette. Ajuste o valor de `optimal_k` na célula abaixo de acordo com o que você observar aqui — o notebook de Araraquara usou K=4, mas em São Paulo, com mais setores e mais heterogeneidade, o número ideal pode ser diferente.

## 11. Modelo K-Means Final

In [ ]:
df_normalized = pd.read_csv("df_rates_normalized_sp.csv", sep=";")

feature_cols = [c for c in df_normalized.columns if c.endswith("_P")]
X = df_normalized[feature_cols].fillna(0)

optimal_k = 4  # <-- ajuste aqui conforme o gráfico do Elbow/Silhouette acima

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X)

df_final = df_normalized.copy()
df_final["cluster"] = clusters

output_file = "df_census_clusters_sp.csv"
df_final.to_csv(output_file, sep=";", index=False, encoding="utf-8-sig")

print(f"Agrupamento concluído com sucesso para {len(df_final)} setores censitários!")
print(f"Arquivo salvo em: {output_file}\n")

print("Distribuição de setores por cluster:")
print(df_final["cluster"].value_counts().sort_index())

display(df_final.head())


In [ ]:
# Estatísticas descritivas das variáveis usadas no cluster
print(df_final[feature_cols].describe().T)

# Quantos setores têm TODAS as variáveis de precariedade em 0?
print((df_final[feature_cols] == 0).all(axis=1).sum(), "de", len(df_final))


## 12. Visualização 2D — Seleção Automática de Eixos

Assim como em Araraquara: em vez de escolher variáveis arbitrárias, o código calcula a variância das médias entre os clusters para cada variável, e seleciona as duas com maior variância (as que melhor diferenciam os grupos) para os eixos X e Y.


In [ ]:
cluster_means = df_final.groupby("cluster")[feature_cols].mean()
mean_variances = cluster_means.var()

top_features = mean_variances.nlargest(2).index.tolist()
x_var, y_var = top_features[0], top_features[1]

print("Variáveis automaticamente selecionadas para máxima separação entre clusters:")
print(f"X-Axis: {x_var} (Variância: {mean_variances[x_var]:.4f})")
print(f"Y-Axis: {y_var} (Variância: {mean_variances[y_var]:.4f})")

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_final,
    x=x_var,
    y=y_var,
    hue="cluster",
    palette="Set1",
    alpha=0.7,
    s=40,
)

plt.title("Visualização 2D Automática dos Clusters - São Paulo", fontsize=13, fontweight="bold", pad=15)
plt.xlabel(f"{x_var} (Maior variância entre clusters)", fontsize=11)
plt.ylabel(f"{y_var} (Segunda maior variância)", fontsize=11)
plt.legend(title="Cluster", loc="best")
plt.grid(True, linestyle="--", alpha=0.5)

plt.savefig("kmeans_auto_2d_plot_sp.png", dpi=300, bbox_inches="tight")
plt.show()


## 13. Visualização 3D Interativa (Plotly)

In [ ]:
import plotly.express as px

top_features_3d = mean_variances.nlargest(3).index.tolist()
var_x, var_y, var_z = top_features_3d[0], top_features_3d[1], top_features_3d[2]

print(f"Variáveis selecionadas -> X: {var_x} | Y: {var_y} | Z: {var_z}")

df_plot_3d = df_final.copy()
df_plot_3d["Cluster"] = df_plot_3d["cluster"].astype(str)

hover_cols = [c for c in ["cd_setor", "CD_SETOR"] if c in df_plot_3d.columns]

fig = px.scatter_3d(
    df_plot_3d,
    x=var_x,
    y=var_y,
    z=var_z,
    color="Cluster",
    color_discrete_sequence=px.colors.qualitative.Set2,
    title="Visualização 3D dos Clusters Censitários - São Paulo",
    hover_data=hover_cols,
)

fig.update_traces(marker=dict(size=3, opacity=0.6, line=dict(width=0.3, color="White")))
fig.update_layout(margin=dict(l=0, r=0, b=0, t=40), width=900, height=700, legend=dict(title="Clusters"))

fig.show()
fig.write_html("kmeans_3d_plot_sp.html")
print("Gráfico salvo em 'kmeans_3d_plot_sp.html'!")


## 14. Mapa Espacial dos Clusters — São Paulo

Usando `geobr` para baixar a malha de setores censitários e cruzar com os resultados do K-Means. O código do município de São Paulo capital é **3550308**.


In [ ]:
#!pip install geobr geopandas

In [ ]:
import geobr
import geopandas as gpd

print("Baixando malha espacial de setores de SP (Censo 2022)...")
sp_mesh = geobr.read_census_tract(code_tract="SP", year=2022)

# Filtra os setores do município de São Paulo (código IBGE: 3550308)
sp_capital_mesh = sp_mesh[sp_mesh["code_muni"] == 3550308].copy()

# Normaliza o código do setor para 15 dígitos nos dois DataFrames
sp_capital_mesh["cd_setor"] = (
    sp_capital_mesh["code_tract"].astype(str).str.split(".").str[0].str.zfill(15)
)

col_setor_final = "cd_setor" if "cd_setor" in df_final.columns else "CD_SETOR"
df_final[col_setor_final] = (
    df_final[col_setor_final].astype(str).str.split(".").str[0].str.zfill(15)
)

# Merge da malha espacial com os clusters do K-Means
gdf_map = sp_capital_mesh.merge(
    df_final, left_on="cd_setor", right_on=col_setor_final, how="inner"
)

print(f"Setores de São Paulo na malha: {len(sp_capital_mesh)}")
print(f"Setores cruzados com sucesso no mapa: {len(gdf_map)}")

fig, ax = plt.subplots(figsize=(12, 12))
sp_capital_mesh.plot(ax=ax, color="#E0E0E0", edgecolor="white", linewidth=0.1)
gdf_map.plot(
    column="cluster",
    categorical=True,
    legend=True,
    ax=ax,
    cmap="Set2",
    edgecolor="white",
    linewidth=0.1,
    legend_kwds={"title": "Cluster", "loc": "lower right"},
)
ax.set_title("Distribuição Espacial dos Clusters Censitários\nSão Paulo - Censo 2022", fontsize=14, fontweight="bold", pad=15)
ax.set_axis_off()
fig.tight_layout()

output_map = "census_clusters_spatial_map_sp.png"
plt.savefig(output_map, dpi=300, bbox_inches="tight")
print(f"Mapa salvo em: {output_map}")
plt.show()


## 15. Perfil dos Clusters — Tabela de Características

Tabela final com a média de cada variável por cluster, traduzida para nomes legíveis e ordenada pela disparidade entre os grupos (quanto maior o desvio-padrão entre clusters, mais essa variável diferencia os grupos).


In [ ]:
legend_dict_en = {
    "V00001_P": "OPPD - Proportion of Occupied Permanent Private Dwellings",
    "v0007_P": "Total - Occupied Private Dwellings (OPPD + OPID)",
    "V00005_P": "OPPD - Average number of residents per dwelling",
    "V00008_P": "OPPD - Proportion of children aged 0 to 9",
    "V00006_P": "OPID - Average number of residents per dwelling",
    "V00009_P": "OPID - Proportion of children aged 0 to 9",
    "V00050_P": "OPPD - Dwelling type: Rooming house or tenement (cortiço)",
    "V00051_P": "OPPD - Dwelling type: Shack or shanty",
    "V00052_P": "OPPD - Dwelling type: Degraded or unfinished residential structure",
    "V00053_P": "OPID - Dwelling type: Tent or canvas, plastic, or fabric shack",
    "V00054_P": "OPID - Dwelling type: Inside an operating commercial establishment",
    "V00055_P": "OPID - Dwelling type: Natural shelter and other improvised structures",
    "V00056_P": "OPID - Dwelling type: Improvised structure in public spaces/streets",
    "V00057_P": "OPID - Dwelling type: Degraded or unfinished non-residential structure",
    "V00058_P": "OPID - Dwelling type: Improvised vehicle",
    "V00113_P": "OPPD - Water: Uses shallow well, water table, or cistern",
    "V00114_P": "OPPD - Water: Uses natural spring, source, or mine",
    "V00115_P": "OPPD - Water: Uses water tank truck (carro-pipa)",
    "V00116_P": "OPPD - Water: Uses stored rainwater",
    "V00117_P": "OPPD - Water: Uses rivers, lakes, streams, or ponds",
    "V00118_P": "OPPD - Water: Uses other forms of water supply",
    "V00200_P": "OPPD - Water: Piped water, but only to the property/land",
    "V00201_P": "OPPD - Water: No piped water supply directly to the dwelling",
    "V00236_P": "OPPD - Sanitation: Shared use bathroom/toilet facility",
    "V00237_P": "OPPD - Sanitation: Basic pit latrine or makeshift toilet structure",
    "V00312_P": "OPPD - Sewage: Discharged into a rudimentary pit or cesspool",
    "V00313_P": "OPPD - Sewage: Discharged into an open ditch",
    "V00314_P": "OPPD - Sewage: Discharged directly into rivers, lakes, streams, or sea",
    "V00315_P": "OPPD - Sewage: Other forms of wastewater disposal",
    "V00316_P": "OPPD - Sewage: Inexistent (no bathroom or toilet facility)",
    "V00399_P": "OPPD - Waste: Burned within the property",
    "V00400_P": "OPPD - Waste: Buried within the property",
    "V00401_P": "OPPD - Waste: Disposed of in vacant lots, hillsides, or public areas",
    "V00402_P": "OPPD - Waste: Other destination given to waste",
    "V00464_P": "OPPD - Water: No connection to the general water distribution network",
    "N_POP_DPPO_P": "Proportion of Resident Population in Favelas / Informal Settlements",
}

present_rates = [c for c in df_final.columns if c in legend_dict_en]

df_characteristics = df_final.groupby("cluster")[present_rates].mean().T

num_clusters = df_final["cluster"].nunique()
column_names = [f"Cluster_{i}" for i in range(num_clusters)]
df_characteristics.columns = column_names

df_characteristics["Intergroup_Disparity"] = df_characteristics[column_names].std(axis=1)
df_characteristics = df_characteristics.sort_values(by="Intergroup_Disparity", ascending=False)

df_characteristics_translated = df_characteristics.rename(index=legend_dict_en)

resident_columns_names = [
    "OPPD - Average number of residents per dwelling",
    "OPID - Average number of residents per dwelling"
]

format_dict = {}
for translated_name in df_characteristics_translated.index:
    if any(r in translated_name for r in resident_columns_names):
        format_dict[translated_name] = "{:.2f} residents"
    else:
        format_dict[translated_name] = "{:.2f}%"

print("=== CLUSTER PROFILE CHARACTERISTICS (São Paulo) ===")

stylized_table = (
    df_characteristics_translated
    .style.format(format_dict)
    .background_gradient(cmap="Blues", axis=1, subset=column_names)
)

display(stylized_table)


---
### Próximos passos sugeridos

1. Ajuste `optimal_k` na célula 11 com base no gráfico de Elbow/Silhouette
2. Observe se alguma variável ficou 100% zerada mesmo em São Paulo (célula da seção 8) — isso indicaria um tipo de precariedade extremamente raro/ausente até na capital
3. Compare o perfil dos clusters de São Paulo com os perfis obtidos na análise da RMSP (nível de município) — isso deve revelar se os padrões de vulnerabilidade dentro da capital seguem lógica territorial semelhante à observada entre os municípios da região metropolitana
